# 06 · Robustness

**Question:** does the headline (H0 shift, Model 3, analysis half: next shot 1.50 ft farther after a
make than after a miss) survive deliberate attempts to break it?

Survival rule and counting (D-045, D-046) were written by Massimo before this notebook ran. The
verdict on what counts as fragile is his.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hothand import core, robustness as rb
from hothand.config import FIGURES, PROCESSED, load_config
from hothand.features import CLUSTERS, FEATURES, SHOTS
from hothand.segments import PLAYER_SEASONS

pd.set_option("display.max_columns", 40, "display.width", 220, "display.max_colwidth", 70)
cfg = load_config()
print("Python:", sys.executable)
print("pandas:", pd.__version__)
HALF = "analysis"
f = pd.read_parquet(FEATURES)
shots = pd.read_parquet(SHOTS)
ps, cl = pd.read_csv(PLAYER_SEASONS), pd.read_csv(CLUSTERS)

## 1. Every variation (one change at a time, Model 3)

In [ ]:
table = rb.run_variations(f, shots, ps, cl, cfg, HALF, cfg["robustness"]["bootstrap_draws"])
baseline = table.loc[table.family == "baseline", "estimate"].iloc[0]
table["change_from_baseline"] = table.estimate - baseline
table["contains_baseline"] = (table.ci_low <= baseline) & (baseline <= table.ci_high)
print(f"Baseline (Model 3, {HALF} half): {baseline:.3f} ft")
table[["family", "variation", "counted", "estimate", "ci_low", "ci_high", "change_from_baseline",
       "contains_baseline", "n_pairs", "n_players"]].round(3)

## 2. Survival rule (D-045): automated check

In [ ]:
check = rb.survival(table, baseline, cfg["robustness"]["survival_share"])
pd.Series(check)

## 3. Specification curve (playbook 7.4)

In [ ]:
c = table[table.counted].sort_values("estimate").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 6.5))
y = np.arange(len(c))
colors = np.where(c.contains_baseline, "#1f77b4", "#d95f02")
ax.errorbar(c.estimate, y, xerr=[c.estimate - c.ci_low, c.ci_high - c.estimate], fmt="none",
            ecolor=colors, capsize=2)
ax.scatter(c.estimate, y, c=colors, s=18, zorder=3)
ax.axvline(baseline, color="#1f77b4", ls="--", lw=1, label=f"baseline {baseline:.2f} ft")
ax.axvline(0, color="#999", lw=1)
ax.set_yticks(y, [f"{a}: {b}" for a, b in zip(c.family, c.variation)], fontsize=7.5)
ax.set_xlabel("After a make minus after a miss: next-shot distance vs own baseline (ft, 95% CI)",
              fontsize=9)
ax.set_title("Specification curve: the headline under every counted variation\n"
             "(orange = interval does not contain the baseline)", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / "stage7_specification_curve.png", dpi=150)

## 4. H2 and H3 gaps under the same variations (reported without a verdict, D-046)

In [ ]:
table.loc[table.h2_low_minus_high.notna(), ["family", "variation", "h2_low_minus_high", "h2_ci_low",
           "h2_ci_high", "h3_rimbigs_minus_3pt", "h3_ci_low", "h3_ci_high"]].round(3)

## 5. Not counted: other clusterings for H3 (D-046)

In [ ]:
clusters_alt = rb.cluster_variants(f, shots, cfg, HALF)
clusters_alt.round(5)

## 6. Placebo tests (playbook 7.3)

**Future-result placebo (D-048):** the shot after the next one cannot influence where the next shot
was taken, so a clear effect here would signal confounding in the design. **Shuffled-order placebo:**
the Stage 5 permutation test already shows the observed shift against 1,000 shuffles
(`reports/figures/stage5_permutation_null.png`).

In [ ]:
d = core.prepare(f, cfg, HALF)
placebo = rb.future_placebo(d)
pd.Series(placebo).round(4)

## 7. Save

In [ ]:
placebo_row = pd.DataFrame([{"half": HALF, "family": "placebo",
                             "variation": "future result (shot after the next one made)",
                             "counted": False, **placebo}])
check_row = pd.DataFrame([{"half": HALF, "family": "survival check (automated)",
                           "variation": "D-045", "counted": False,
                           "estimate": check["share_containing_baseline"],
                           "note": f"all positive: {check['all_positive']}; rule met: {check['rule_met']}"}])
clusters_alt.insert(0, "half", HALF)
out = pd.concat([table, clusters_alt, placebo_row, check_row], ignore_index=True)
out.to_csv(PROCESSED / "results_robustness.csv", index=False)
print(f"{len(out)} rows -> data/processed/results_robustness.csv")